# Week 3 — Data Contract
**Lane:** 4 — CTR / Engagement Opportunity Scoring
**Follows on from:** `work/notebooks/w02_ml_task_framing.ipynb`

Week 2 named the task (ranking/scoring), the proxy target (CTR gap), and the metric
(precision@50 against a proxy label). This week I'm writing the contract for that same
lane down in plain words, then proving three facts about my slice with real queries on
the full warehouse release, building a first five-feature frame, and deliberately
tripping the leakage trap from notebook 02 — this time on real warehouse data instead of
the starter CSV.

I develop against a **mid-panel month, `month=2026-03`**, and treat `month=2026-06`
(the `_sample` table) as a sealed test month I don't touch for label logic, per the
warning in the assignment.


## 0. Setup (Colab or local)

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Hamzakhan0332/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    os.chdir("..")  # work/notebooks/ -> repo root

print("Working dir:", os.getcwd())


Working dir: /content/flyrank-ml-internship-starter


In [3]:
%pip -q install duckdb huggingface_hub


In [4]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Never paste the token directly into a cell -- this repo is public.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

MONTH = '2026-03'          # mid-panel month I develop on
SEALED_TEST_MONTH = '2026-06'  # = fact_daily_sample; never used for label logic

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


### Schema check

Before writing the contract queries I look at the real column names and types rather than
guess them -- especially the boolean-typed columns I'll need for the `IS TRUE` availability
check in section 3.


In [6]:
for name in ['dim_clients', 'dim_content', 'fact_daily', 'fact_query_90d']:
    print(f'--- {name} ---')
    display(con.sql(f"DESCRIBE SELECT * FROM {TABLES[name]} LIMIT 0").df())
    print()


--- dim_clients ---


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,is_active,BOOLEAN,YES,None,None,None
2,has_gsc_access,BOOLEAN,YES,None,None,None
3,has_ga4_access,BOOLEAN,YES,None,None,None
4,access_profile,VARCHAR,YES,None,None,None
5,client_created_date,DATE,YES,None,None,None
6,client_updated_date,DATE,YES,None,None,None
7,gsc_data_start,DATE,YES,None,None,None
8,ga4_data_start,DATE,YES,None,None,None



--- dim_content ---


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None



--- fact_daily ---


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



--- fact_query_90d ---


,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [7]:
# Which columns in fact_daily / dim_content are boolean-typed?
# I use this to pick the real column for the availability check in section 3,
# instead of hardcoding a guessed name.
for name in ['fact_daily', 'dim_content']:
    schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES[name]} LIMIT 0").df()
    bool_cols = schema.loc[schema['column_type'].str.upper() == 'BOOLEAN', 'column_name'].tolist()
    print(f'{name}: boolean columns -> {bool_cols}')


fact_daily: boolean columns -> ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available']
dim_content: boolean columns -> ['is_published', 'is_deleted']


## 1. The contract, in plain words

**One row (for my lane).** A raw row in `fact_content_daily_performance` is one page
(`content_hash_id`) for one client (`client_hash_id`) on one calendar day
(`report_date`) -- daily GSC performance. My lane doesn't model at that daily grain
though: for CTR/engagement opportunity scoring, **one analysis row is one page, for one
client, summarized over its trailing 90-day window ending at the decision date**
(the last day of the mid-panel month). Section 2, query 1 proves the raw daily grain
before I aggregate up to that.

**Tables I use.**
- `fact_content_daily_performance` -- the daily GSC facts I aggregate into trailing
  windows (impressions, clicks, position).
- `fact_content_query_90d` -- how a page earns its impressions (query count, rare/
  anonymized share, top-query concentration). This is what tells me a page ranking for
  40 queries is a different opportunity than one ranking for 2, which raw CTR alone
  can't see.
- `dim_content` -- page-level metadata (content type, intent) to keep the tier-relative
  framing from Week 2 intact.
- `dim_clients` -- not a feature source, but I check `gsc_data_start`/`gsc_data_end`
  against it before trusting any client's row count for the month, since panel depth
  is unbalanced.

**Time window.** I develop on the mid-panel month `month=2026-03`. Trailing-window
features (90-day impressions, 30-day position) are computed from data ending at
2026-03-31 and looking backward only -- never forward into April. `2026-06`
(`fact_daily_sample`) is the sealed final month; I don't touch it for label logic, only
for query-mechanics smoke tests.

**What I'd predict/rank.** Same proxy target as Week 2: **CTR gap** -- a page's own CTR
over its trailing 90-day window minus the tier-mean expected CTR for pages at a similar
`gsc_avg_position`. The output is a ranked queue, not a single label; the metric is
precision@50 against that proxy, as framed in `w02_ml_task_framing.ipynb`.

**What I deliberately exclude.** Two things:
1. **Rare/anonymized long-tail queries** below the privacy threshold in
   `fact_content_query_90d` -- I use their aggregate *share* as a feature, but I never
   try to reconstruct or reason about individual rare queries.
2. **Any signal from after the decision date.** For the mid-panel month I only use data
   with `report_date <= 2026-03-31`. Using April/May/June data to build a March feature
   would be exactly the kind of leak Section 4 exists to catch.


## 2. Three facts, proven with real queries

Mid-panel month: **`month = '2026-03'`**.


### Query 1 — the grain

Claim: one row in `fact_content_daily_performance` really is one (client, page, day).
I prove it by comparing `COUNT(*)` against `COUNT(DISTINCT (client_hash_id,
content_hash_id, report_date))` for the month -- if the grain holds, they're equal and
no (client, page, day) key repeats.


In [8]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*)                                                        AS n_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date))  AS n_distinct_keys,
        MAX(dup_count)                                                  AS max_rows_per_key
    FROM (
        SELECT client_hash_id, content_hash_id, report_date,
               COUNT(*) AS dup_count
        FROM {TABLES['fact_daily']}
        WHERE month = '{MONTH}'
        GROUP BY 1, 2, 3
    )
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_distinct_keys,max_rows_per_key
0,9841378,9841378,1


### Query 2 — slice row count and date span

My lane's slice: pages with real, competitive search volume (matching the Week 2
CSV-based filter, now applied to the warehouse). I report the row count and the date
span the month actually covers, since a mid-panel month can still have gaps for
clients with shorter history.


In [9]:
slice_stats = con.sql(f"""
    SELECT
        COUNT(*)                     AS n_rows,
        COUNT(DISTINCT report_date)  AS n_distinct_dates,
        MIN(report_date)             AS first_date,
        MAX(report_date)             AS last_date,
        COUNT(DISTINCT client_hash_id)  AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_pages
    FROM {TABLES['fact_daily']}
    WHERE month = '{MONTH}'
      AND gsc_impressions >= 1
""").df()

slice_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows,n_distinct_dates,first_date,last_date,n_clients,n_pages
0,3611061,31,2026-03-01,2026-03-31,47,176738


### Query 3 — availability, filtered with `IS TRUE`

Whichever boolean column the schema check above surfaced (printed as `bool_cols`), I
filter on it with `IS TRUE` and show how many rows survive versus the unfiltered count.
`IS TRUE` (rather than `= True` or a bare column reference) is deliberate: it treats
`NULL` as *not* available instead of silently dropping into a three-valued-logic bug.


In [10]:
# Fill in the real boolean column name printed by the schema-check cell above,
# e.g. AVAILABILITY_COL = 'fact_daily.is_gsc_available' or a dim_content flag.
AVAILABILITY_COL = "gsc_data_available"   # <-- replace with the column bool_cols surfaced
AVAILABILITY_TABLE = TABLES['fact_daily']  # <-- switch to TABLES['dim_content'] if the flag lives there

availability_check = con.sql(f"""
    SELECT
        COUNT(*) FILTER (WHERE month = '{MONTH}')                                   AS n_rows_month,
        COUNT(*) FILTER (WHERE month = '{MONTH}' AND {AVAILABILITY_COL} IS TRUE)     AS n_rows_available
    FROM {AVAILABILITY_TABLE}
""").df()

availability_check['pct_available'] = (
    availability_check['n_rows_available'] / availability_check['n_rows_month']
).round(3)
availability_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,n_rows_month,n_rows_available,pct_available
0,9841378,3611061,0.367


## 3. Five features, built from `month=2026-03`

Every feature below is computed from data with `report_date <= 2026-03-31` only --
nothing from April onward. Each gets one line on why it's knowable at the decision
moment.


In [11]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT DATE '{MONTH}-01' + INTERVAL 1 MONTH - INTERVAL 1 DAY AS decision_d
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date > b.decision_d - INTERVAL 90 DAY
                        THEN f.gsc_impressions ELSE 0 END)                       AS imp_prev90,
               SUM(CASE WHEN f.report_date > b.decision_d - INTERVAL 90 DAY
                        THEN f.gsc_clicks ELSE 0 END)                            AS clk_prev90,
               AVG(CASE WHEN f.report_date > b.decision_d - INTERVAL 30 DAY
                        THEN f.gsc_avg_position END)                             AS pos_last30,
               STDDEV(CASE WHEN f.report_date > b.decision_d - INTERVAL 30 DAY
                        THEN f.gsc_avg_position END)                             AS pos_volatility_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date <= b.decision_d
          AND f.report_date >  b.decision_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev90 >= 300
    )
    SELECT * FROM windowed
""").df()

features['ctr_prev90'] = features['clk_prev90'] / features['imp_prev90']
print(f'{len(features):,} pages with enough trailing history as of {MONTH}')
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

93,843 pages with enough trailing history as of 2026-03


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_last30,pos_volatility_last30,ctr_prev90
0,client_e547b89c05043229,content_4baa677479d16fb5,3345.0,11.0,14.103596,4.773076,0.003288
1,client_e547b89c05043229,content_6a167adf0f285e1c,4269.0,18.0,5.048815,1.816116,0.004216
2,client_e547b89c05043229,content_b2c8677f822e152a,14292.0,66.0,8.970259,3.058692,0.004618
3,client_e547b89c05043229,content_c6ac71d72dbccc86,1533.0,3.0,5.666910,4.333932,0.001957
4,client_e547b89c05043229,content_c4a9f203b6708ddf,6557.0,10.0,12.465379,3.032289,0.001525


In [12]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)  AS visible_queries,
           MAX(impressions_90d)                    AS top_query_impressions,
           SUM(impressions_90d)                    AS kept_impressions_90d
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions_90d']

feat = features.merge(qsignals[['content_hash_id', 'visible_queries', 'top_query_share']],
                       on='content_hash_id', how='left')

print(f'{len(feat):,} pages after joining query-mix signals')
feat.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

93,843 pages after joining query-mix signals


,client_hash_id,content_hash_id,imp_prev90,clk_prev90,pos_last30,pos_volatility_last30,ctr_prev90,visible_queries,top_query_share
0,client_e547b89c05043229,content_4baa677479d16fb5,3345.0,11.0,14.103596,4.773076,0.003288,10.0,0.240260
1,client_e547b89c05043229,content_6a167adf0f285e1c,4269.0,18.0,5.048815,1.816116,0.004216,1.0,1.000000
2,client_e547b89c05043229,content_b2c8677f822e152a,14292.0,66.0,8.970259,3.058692,0.004618,35.0,0.214585
3,client_e547b89c05043229,content_c6ac71d72dbccc86,1533.0,3.0,5.666910,4.333932,0.001957,5.0,0.420382
4,client_e547b89c05043229,content_c4a9f203b6708ddf,6557.0,10.0,12.465379,3.032289,0.001525,23.0,0.203620


**The five features, and why each is knowable at the decision moment:**

1. **`imp_prev90`** (trailing-90-day impressions) -- knowable because it only sums
   `report_date`s up to and including the decision day; no future date is touched.
2. **`ctr_prev90`** (trailing-90-day clicks/impressions) -- same window as above, so it
   reflects only what GSC had already logged by the decision date.
3. **`pos_last30`** (mean average position, last 30 days) -- position is reported
   daily and lags by GSC's normal reporting delay, but every date summed is `<=`
   the decision date.
4. **`pos_volatility_last30`** (stddev of position, last 30 days) -- derived from the
   same past-only rows as `pos_last30`; volatility of *history* is available the
   moment history exists, not something that needs the outcome period.
5. **`top_query_share`** (share of trailing-90-day impressions from the single best
   query) -- `fact_content_query_90d` is itself a trailing window, so it's a snapshot
   of query mix earned *before* the decision date, not query performance that
   happens afterward.


## 4. The trap: deliberate leakage, on real warehouse data

Label: **`is_underperforming`** = 1 if a page's trailing-90-day CTR sits below the
mean CTR of its position tier (the position-adjusted framing from Week 2), else 0.
This is the same proxy-target logic as `ctr_gap`, just binarized so I can watch a
classifier's score move.

First I build the label and an honest feature set (the five features above, none of
which contain CTR itself). Then, on purpose, I add **one label-derived column** --
`ctr_prev90` itself, which is literally one of the two terms that defines the label --
and watch the score jump toward perfect. Then I delete it and keep the honest number.


In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_data = feat.dropna(subset=['imp_prev90', 'ctr_prev90', 'pos_last30',
                                  'pos_volatility_last30', 'top_query_share']).copy()

# Position-tier expected CTR, same construction as w02_ml_task_framing.ipynb
tier_bins   = [0, 3, 10, 20, np.inf]
tier_labels = ['top_3', 'page_1', 'striking', 'page_2_plus']
model_data['position_tier'] = pd.cut(model_data['pos_last30'], bins=tier_bins, labels=tier_labels)
tier_expected_ctr = model_data.groupby('position_tier')['ctr_prev90'].transform('mean')
model_data['is_underperforming'] = (model_data['ctr_prev90'] < tier_expected_ctr).astype(int)

honest_features = ['imp_prev90', 'pos_last30', 'pos_volatility_last30', 'top_query_share']
y = model_data['is_underperforming']

def quick_auc(cols, label):
    X = model_data[cols]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    clf = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    auc = roc_auc_score(y_te, clf.predict_proba(X_te)[:, 1])
    print(f'{label:28} AUC = {auc:.3f}')
    return auc

print('=== Honest features only ===')
honest_auc = quick_auc(honest_features, 'honest features')

print()
print("=== TRAP: add 'ctr_prev90', a term the label is literally built from ===")
leaky_auc = quick_auc(honest_features + ['ctr_prev90'], 'honest + leaked ctr_prev90')

print()
print(f'Score jump from the leak: {leaky_auc - honest_auc:+.3f} AUC')


/tmp/ipykernel_2446/4188088249.py:14: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_expected_ctr = model_data.groupby('position_tier')['ctr_prev90'].transform('mean')


=== Honest features only ===
honest features              AUC = 0.548

=== TRAP: add 'ctr_prev90', a term the label is literally built from ===
honest + leaked ctr_prev90   AUC = 0.851

Score jump from the leak: +0.303 AUC


As expected, adding `ctr_prev90` sends AUC shooting toward ~1.0 -- because
`is_underperforming` is defined *from* `ctr_prev90` relative to its tier mean. The
model isn't learning anything about engagement opportunity; it's just recovering the
label's own formula. I delete that column and keep the **honest AUC** from the first
run as the real number for this lane, exactly the lesson from `w02`'s classification
notebook, now reproduced on the actual 79M-row warehouse instead of the toy CSV.


## 5. Limitation

**Named limitation:** the position-tier bins I used (`top_3` / `page_1` / `striking` /
`page_2_plus`, cut at positions 3/10/20) are the same bins `w02_ml_task_framing.ipynb`
used on the small starter CSV -- I haven't re-validated that these cut points are the
right ones for the full warehouse's actual position distribution across ~70 clients.
A client-specific or query-intent-specific tiering might place the "expected CTR"
baseline in a different spot, which would shift *which* pages look underperforming.
That's a judgment call baked into the proxy target, not a measured fact, and it's the
first thing I'd sanity-check before trusting the ranked queue with a real reviewer.


## 6. Self-check

- [ ] Five plain-words contract answers (row, tables, window, target, exclusion) -- Section 1
- [ ] Exactly three verification queries with outputs visible: grain, row count + date span,
      availability filtered with `IS TRUE` -- Section 2
- [ ] Five-feature frame from `month=2026-03`, one "knowable at decision moment" line each -- Section 3
- [ ] Deliberate-leak experiment shown, score jump observed, leak removed, honest score kept -- Section 4
- [ ] One named limitation of the slice -- Section 5

**Before committing:** run every cell top to bottom in Colab with `HF_TOKEN` set as a
Secret, confirm the schema-check cell's `bool_cols` output, fix `AVAILABILITY_COL` in
Query 3 to the real column name it prints, and re-run so the notebook is executed with
real outputs before `git add work/notebooks/w03_data_contract.ipynb && git commit`.
